# Lab 11 Fourier spectral method

#### <p style="text-align: right;"> &#9989; **put your name here** </p>

# Due 12/1
---
In this coding assignment, we will use Fourier spectal method to simulate diffusion in a 2D domain. First we will solve a 2D diffusion equation using finite difference method.

$$\frac{\partial C}{\partial t} = D \bigg(\frac{\partial^2 C}{\partial x^2} + \frac{\partial^2 C}{\partial y^2}\bigg).$$

The 5-point stencil is given as

$$\frac{C_{i,j}^{(n+1)}-C_{i,j}^{(n)}}{\Delta t} = D\frac{C_{i,j-1}^{(n)}+C_{i,j+1}^{(n)}+C_{i-1,j}^{(n)}+C_{i+1,j}^{(n)}-4C_{i,j}^{(n)}}{\Delta x^2}.$$

Run the cell below for setting up the simulation initial condition.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

# -----------------------
# define domain and discretization
# -----------------------
Ly = 128
py = 128
y = np.linspace(0, Ly , py, endpoint=False)

Lx = 128
px = 128
x = np.linspace(0, Lx, px, endpoint=False)

dx = Lx/px
dt = 0.01
Nstep = 5001 

D = 1.0

C = np.zeros((py,px))
Lap = np.zeros((py,px))

# -- initial condition
for i in range(py):
    for j in range(px):
        # rad = np.sqrt((i-py/2)**2 + (j-px/2)**2)
        # C[i,j] = np.exp(-0.005*rad**2)
        if (i > 38 and i < 90) and (j > 32 and j < 96):
            C[i,j] = 1.0

# -- visualization
fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(C, origin='upper', interpolation='nearest', cmap='viridis')  
cbar = fig.colorbar(im, ax=ax, label='phi')            # <- colorbar once
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(0, Lx)
ax.set_ylim(0, Ly)        

The cell below runs the simulation of 2D diffusion.

In [ ]:
## -- time evolution
for it in range(Nstep):

    # Laplace on the RHS (nest loop)
    # for i in range(1,py-1):
    #     for j in range(1,px-1):
    #         Lap[i,j] = (C[i,j-1] + C[i,j+1] + C[i-1,j] + C[i+1,j] - 4*C[i,j])/dx**2
        
    # vectorized calcualtion of Laplace (vectorized)
    Lap[1:-1,1:-1] = (C[1:-1,0:-2] + C[1:-1,2:] + C[0:-2,1:-1] + C[2:,1:-1] 
                      - 4*C[1:-1,1:-1])/dx**2

    #Euler time method
    C[1:-1,1:-1] = C[1:-1,1:-1] + dt*D*Lap[1:-1,1:-1]

    # periodic boundary conditions
    C[0,  1:-1] = C[-2, 1:-1]
    C[-1, 1:-1] = C[1,  1:-1]
    C[:,  0   ] = C[:,  -2  ]
    C[:, -1   ] = C[:,  1   ]

    # visualization
    if it % 100 == 0:
        im.set_data(C)                 # update image only
        ax.set_title(f"{it}")
        
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        # fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation     

---
## Fourier spectral method

In the 1D example of Fourier transform of a 2nd derivative, we had

$$F(f''(x)) = - \omega^2 F(f(x)).$$

In 2D, the Fourier transform of 2nd order derivative is

$$F(\nabla^2 f(x,y)) = -\big(w_x^2 + w_y^2 \big) F(f(x,y)).$$

Note that we have wave numbers in the $x$ and $y$ directions. We will use the cell below to set up the reciprocal basis for 2D problems.

In [ ]:
# -----------------------
# reciprocal basis in x and y direction
# -----------------------
wiy = ??
wix = ??
# squares of wave numbers
wix2 = wix**2       # shape: (px,)
wiy2 = wiy**2       # shape: (py,)
# build k^2 = ky^2 + kx^2 on the 2D grid using broadcasting
Wi2 = wix2[None, :] + kwi2[:, None]


# fftshift
wy = ??
wx = ??

# squares of wave numbers
wy2 = wy**2          
wx2 = wx**2          

# build k^2 = ky^2 + kx^2 on the 2D grid using broadcasting
W2 = ??  # shape: (py, px)

fig = plt.figure(figsize=(8,3))

# --- Left subplot ---
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(kox[None, :], koy[:, None], Ko2, cmap='viridis')
ax1.set_title("using original reciprocal basis")

# --- Right subplot ---
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(kox[None, :], koy[:, None], K2, cmap='plasma')
ax2.set_title("after fftshift")

plt.tight_layout()
plt.show()

&#9989;  Do This - Based on your observation of the $w^2$, how the reciprocal bases are swapped in terms of the 4 quadrons on the $x$-$y$ plane?  

---
Set up the initial condition

In [ ]:
# -- initial condition
for i in range(py):
    for j in range(px):
        if (i > 38 and i < 90) and (j > 32 and j < 96):
            C[i,j] = 1.0

# -- visualization
fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(C, origin='upper', interpolation='nearest', cmap='viridis')  
cbar = fig.colorbar(im, ax=ax, label='phi')            # <- colorbar once
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(0, Lx)
ax.set_ylim(0, Ly)              

---
Simulation of diffusion using Fourier spectral method. For 2D FFT, we use `fft2` function. Note that we can use vectorized computing as in the case of finite difference method. We will need to use a 2D array of the square of 2D wave numbers, which is the W2 in the cell eariler.

Take discrete Fourier transform of the diffusion equation:

$$F\bigg[\frac{C^{(n+1)}-C^{(n)}}{\Delta t} \bigg] = F\big[ D \nabla^2 C^{(n)} \big] \Longrightarrow \frac{F\big[C^{(n+1)}\big]-F\big[C^{(n)}\big]}{\Delta t}= -\omega^2 D F\big[C^{(n)} \big].$$

The time step becomes

$$F\big[C^{(n+1)} \big] = F\big[C^{(n)} \big] + \Delta t \times \big\{-\omega^2 D F\big[ C^{(n)} \big]\big\} = F\big[C^{(n)} \big] - \Delta t \omega^2 D F\big[ C^{(n)} \big].$$

In [ ]:
# container for C in Fourier space
CnFt = np.fft.fft2(??)

# pre-allocate container for Laplace of C in Fourier space
LpFt = np.zeros_like(CnFt, dtype=complex)

# -- time evolution
for it in range(0, Nstep):

    # compute Laplacian in Fourier space
    # for p in range(py):
    #     for q in range(px):
    #         LpFt[p, q] = -(wy[p]**2 + wx[q]**2) * CnFt[p, q]

    # vectorized Laplacian in Fourier space
    LpFt = ??

    # time update
    CnFt = ??


    # visualization
    if it % 100 == 0:
        C = np.fft.ifft2(CnFt)
        im.set_data(np.real(C))                 # update image only
        ax.set_title(f"{it}")
        
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        # fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation 

---
&#9989; Do This - Do you impose boundary conditions when using Fourier spectral method? What kind of boundary conditions are used in Fourier method? 

&#9989; Do This - Can you impose Dirichlet or Neumann boundary conditions in Fourier spectral method?

&#9989; Do This - List at least one other applications of FFT.

### Great! You're done. Please upload your completed file to the drop box on the course webpage.